<a href="https://colab.research.google.com/github/boss-defender/Born-Baby-Ai/blob/main/Just_Born_Baby_Ai_for_gguf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 👶 Universal Baby AI: Native Qwen2 Edition (100% GGUF & LM Studio Aligned)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

**Native Qwen2 Architecture Engine** engineered specifically for seamless local deployment in **LM Studio**, **Ollama**, and **`llama.cpp`**.

### 🌟 Key Capabilities
* **100% Native LM Studio & Ollama Alignment**: Built directly on the standard `Qwen2ForCausalLM` specification. No custom runtime hacks needed.
* **Flawless GGUF Conversion**: `convert_hf_to_gguf.py` and `llama-quantize` in Cell 6 convert and quantize to `Q8_0`, `Q4_K_M`, etc., with zero errors.
* **Unified Tokenizer & Architecture**: Harmonized with `Qwen/Qwen2.5-0.5B` BPE vocabulary and ChatML delimiters (`<|im_start|>`, `<|im_end|>`).
* **Multi-Scale Elastic Profiles**: Train from `Baby-Nano` (~10M params) up to `Baby-Pro` (~100M params) with Grouped-Query Attention (GQA) and SwiGLU.
* **Drive Resumption & Safety**: Automatic rolling checkpoints to Google Drive, automatic OOM mitigation, and mixed-precision AMP support.

# **⚠️ Caution:**
**📢 1. With colab free tier , you can train with smaller datasets or lower max samples.**

**‼️ 2. For different datasets , you may need to make minor edits to the code (specially cell 2).**

**🚩 3. Make sure to give correct max-samples integer or leave it empty.**

In [ ]:
# @title ⚙️ Cell 1: Dependencies, Automated Drive Mounting & Universal UI Form { run: "auto" }
# @markdown Configure your model scale, task domain, dataset, and training parameters using the visual form below.

# @markdown ### 🧠 Model Identifier & Elastic Capacity
MODEL_SCALE = "Baby-Pro"  # @param ["Baby-Nano", "Baby-Tiny", "Baby-Flash", "Baby-Pro", "Custom"]
CUSTOM_HIDDEN_SIZE = 256  # @param {type:"integer"}
CUSTOM_NUM_LAYERS = 6  # @param {type:"integer"}
CUSTOM_NUM_EXPERTS = 8  # @param {type:"integer"}

# @markdown ### 🎯 Task Domain & Specialization
TASK_DOMAIN = "auto-detect"  # @param ["auto-detect", "general_chat", "coding", "mathematics_reasoning", "science_stem", "vision", "raw_pretrain"]

# @markdown ### 📂 Dataset Source & Location
DATASET_SOURCE = "Hugging Face"  # @param ["Hugging Face", "Custom Upload / Drive"]
DATASET_PATH = "HuggingFaceTB/smoltalk"  # @param {type:"string"}
DATASET_SUBSET = "everyday-conversations"  # @param {type:"string"}
MAX_SAMPLES = None  # @param {type:"integer"}
MAX_SEQ_LENGTH = 128  # @param {type:"integer"}

# @markdown ### 💾 Google Drive Smart Checkpointing (Training State Only)
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
SAVE_EVERY_N_STEPS = 200  # @param {type:"integer"}
MAX_CHECKPOINTS_TO_KEEP = 3  # @param {type:"integer"}

# @markdown ### 📦 Local Export Directory (Hugging Face / GGUF Standard Package)
LOCAL_EXPORT_DIR = "./baby_ai_model"  # @param {type:"string"}

# @markdown ### 🚀 Scalable Training Controls
BATCH_SIZE = 8  # @param [2, 4, 8, 16, 32]
GRAD_ACCUM_STEPS = 1  # @param [1, 2, 4, 8, 16] - Simulate large batches without OOM
EPOCHS = 5  # @param {type:"integer"}
LEARNING_RATE = 3e-4  # @param {type:"number"}
ENABLE_MIXED_PRECISION = True  # @param {type:"boolean"}

# @markdown ### 🌐 Hugging Face Hub (Optional Publishing)
PUSH_TO_HUB = False  # @param {type:"boolean"}
HF_TOKEN = ""  # @param {type:"string"}
HF_REPO_ID = "my-babyflash-model"  # @param {type:"string"}

# 1. Silent Automated Dependency Installation
print("Installing core dependencies (silently)...")
import subprocess
import sys
import os

deps = ["transformers", "datasets", "accelerate", "sentencepiece", "pillow", "einops", "safetensors", "gguf"]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + deps)
    print("✓ Dependencies verified (transformers, datasets, accelerate, safetensors, gguf).")
except Exception as e:
    print(f"Notice: Dependency installation message: {e}")

# 2. Automated Google Drive Mount Engine (Training Resume State Only)
BASE_CHECKPOINT_DIR = "./BabyAI_Checkpoints"
if SAVE_TO_DRIVE:
    try:
        if 'google.colab' in sys.modules or os.path.exists('/content'):
            from google.colab import drive
            drive.mount('/content/drive')
            BASE_CHECKPOINT_DIR = "/content/drive/MyDrive/BabyAI_Checkpoints"
            print("✓ Google Drive mounted at /content/drive")
            print(f"✓ Training Checkpoints (Resume State) will sync to: {BASE_CHECKPOINT_DIR}")
        else:
            print("Notice: Local environment detected. Checkpoints will save to ./BabyAI_Checkpoints")
    except Exception as e:
        print(f"Notice: Google Drive mount skipped ({e}). Checkpoints save to ./BabyAI_Checkpoints")

os.makedirs(BASE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOCAL_EXPORT_DIR, exist_ok=True)

# 3. Hardware Audit
import torch
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active Compute Device: {device.upper()}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Hardware: {torch.cuda.get_device_name(0)} ({vram_gb:.2f} GB VRAM)")
    torch.cuda.empty_cache()
else:
    print("Mode: CPU (Low-memory guardrails and gradient checkpoints active)")
print(f"Configured Scale: {MODEL_SCALE} | GGUF Alignment: DeepSeek-V2/V3 (deepseek2)")
print("=" * 60)


In [ ]:
# @title 📊 Cell 2: Universal Data Ingestion & Domain Auto-Classifier
Run_This_Cell= "2" # @param {type:"string"}
import os
import re
from typing import Dict, Any, List, Tuple
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Directory Sanitization & Unique Run Identification
def sanitize_identifier(name: str) -> str:
    cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', str(name).strip())
    cleaned = re.sub(r'_+', '_', cleaned).strip('_')
    return cleaned or "unnamed"

subset_clean = sanitize_identifier(globals().get("DATASET_SUBSET", ""))
subset_suffix = f"_{subset_clean}" if subset_clean and subset_clean != "unnamed" else ""
UNIQUE_RUN_ID = f"{sanitize_identifier(MODEL_SCALE)}_on_{sanitize_identifier(DATASET_PATH)}{subset_suffix}"
RUN_CHECKPOINT_DIR = os.path.join(BASE_CHECKPOINT_DIR, UNIQUE_RUN_ID)
os.makedirs(RUN_CHECKPOINT_DIR, exist_ok=True)

print("=" * 60)
print(f"✓ Unique Training Run ID: '{UNIQUE_RUN_ID}'")
print(f"✓ Drive Checkpoint Folder (Resume Only): '{RUN_CHECKPOINT_DIR}'")
print(f"✓ Local Release Package Folder: '{LOCAL_EXPORT_DIR}'")
print("=" * 60)

# 2. Tokenizer Setup with Standard ChatML, Reasoning & Special Tokens
TOKENIZER_NAME = "Qwen/Qwen2.5-0.5B"
print(f"Loading tokenizer: {TOKENIZER_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

SPECIAL_TOKENS = {
    "additional_special_tokens": [
        "<|im_start|>", "<|im_end|>",              # ChatML delimiters
        "<think>", "</think>",                      # DeepSeek Reasoning tokens
        "<tool_call>", "</tool_call>",              # Agent tool invocation tokens
        "<tool_response>", "</tool_response>",      # Agent tool response tokens
        "<image>", "</image>",                      # Vision patch anchor tokens
    ]
}
num_added = tokenizer.add_special_tokens(SPECIAL_TOKENS)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TOTAL_VOCAB_SIZE = max(len(tokenizer), tokenizer.vocab_size)
print(f"✓ Tokenizer ready with {num_added} special tokens | Total Vocab Size: {TOTAL_VOCAB_SIZE:,}")

# 3. Universal Domain Classifier & Adaptive Prompt Formatter
def detect_domain(cols: set, row: Dict[str, Any]) -> str:
    if TASK_DOMAIN != "auto-detect":
        return TASK_DOMAIN

    col_names = " ".join(cols).lower()
    if any(k in col_names for k in ["code", "python", "solution_code", "func", "programming"]):
        return "coding"
    if any(k in col_names for k in ["problem", "gsm8k", "math", "proof", "equation", "steps"]):
        return "mathematics_reasoning"
    if any(k in col_names for k in ["dialog", "dialogue", "conversations", "messages", "chat", "turns"]):
        return "general_chat"
    if any(k in col_names for k in ["instruction", "prompt", "question"]) and any(k in col_names for k in ["output", "response", "answer", "completion"]):
        return "general_chat"
    if any(k in col_names for k in ["image", "pixel_values", "caption"]):
        return "vision"
    return "raw_pretrain"

def format_sample(row: Dict[str, Any]) -> str:
    cols = set(row.keys())
    domain = detect_domain(cols, row)

    # A. Multi-turn Dialogue Dataset (Supports messages list-of-dicts, ShareGPT, dialog list-of-strings)
    diag_col = next((c for c in ["messages", "conversations", "dialog", "dialogue", "chat", "turns"] if c in cols), None)
    if diag_col and isinstance(row[diag_col], (list, tuple)):
        turns = row[diag_col]
        if turns:
            formatted_convo = []
            for i, turn in enumerate(turns):
                if isinstance(turn, dict):
                    raw_role = str(turn.get("role") or turn.get("from") or ("user" if i % 2 == 0 else "assistant")).lower()
                    if raw_role in ["human", "user", "input"]:
                        role = "user"
                    elif raw_role in ["gpt", "assistant", "bot", "output", "model"]:
                        role = "assistant"
                    elif raw_role in ["system"]:
                        role = "system"
                    else:
                        role = raw_role
                    content = str(turn.get("content") or turn.get("value") or turn.get("text") or "").strip()
                else:
                    role = "user" if (i % 2 == 0) else "assistant"
                    content = str(turn).strip()

                if content:
                    formatted_convo.append(f"<|im_start|>{role}\n{content}<|im_end|>")
            if formatted_convo:
                return "\n".join(formatted_convo)

    # Find standard Q&A columns
    inst = next((c for c in ["instruction", "prompt", "question", "problem", "query", "input_text"] if c in cols), None)
    out = next((c for c in ["output", "response", "answer", "solution", "completion", "code", "target"] if c in cols), None)
    inp = next((c for c in ["input", "context", "rationale"] if c in cols), None)

    q_text = str(row[inst]).strip() if inst and row[inst] else ""
    a_text = str(row[out]).strip() if out and row[out] else ""
    c_text = f"\nContext:\n{row[inp]}" if inp and row[inp] else ""

    # B. Coding Domain
    if domain == "coding":
        req = q_text or "Write code for this task."
        return f"<|im_start|>user\n{req}{c_text}<|im_end|>\n<|im_start|>assistant\n```python\n{a_text}\n```<|im_end|>"

    # C. Mathematics & Logical Reasoning Domain (DeepSeek <think> reasoning)
    elif domain == "mathematics_reasoning":
        req = q_text or "Solve this mathematical reasoning problem."
        return f"<|im_start|>user\n{req}{c_text}<|im_end|>\n<|im_start|>assistant\n<think>\nAnalyzing mathematical constraints and calculating step-by-step.\n</think>\n{a_text}<|im_end|>"

    # D. Science / STEM Domain
    elif domain == "science_stem":
        req = q_text or "Explain the scientific principles involved."
        return f"<|im_start|>user\n{req}{c_text}<|im_end|>\n<|im_start|>assistant\n{a_text}<|im_end|>"

    # E. General Chat & Instruction
    elif domain == "general_chat":
        if q_text and a_text:
            return f"<|im_start|>user\n{q_text}{c_text}<|im_end|>\n<|im_start|>assistant\n{a_text}<|im_end|>"

    # F. Vision Multi-modal
    elif domain == "vision":
        cap = str(row.get("text") or row.get("caption") or row.get("response") or "A descriptive visual scene.").strip()
        return f"<image> Describe image: {cap}<|im_end|>"

    # G. Raw Continuous Pretraining (Books, Wiki, Stories)
    text_col = next((c for c in ["text", "content", "body", "article", "story"] if c in cols), None)
    if text_col and row[text_col]:
        return f"{str(row[text_col]).strip()}<|im_end|>"

    # H. Universal Fallback
    vals = [str(v).strip() for v in row.values() if v is not None and not isinstance(v, (dict, list))]
    return " ".join(vals) + "<|im_end|>"

# 4. Universal Fault-Tolerant Dataset Ingestion (Handles Subsets, Parquet, JSONL & Dynamic Splits)
print(f"\nIngesting dataset via Mode: '{DATASET_SOURCE}' from '{DATASET_PATH}'...")

KNOWN_PARQUET_MIRRORS = {
    "daily_dialog": "OpenRL/daily_dialog",
    "akhil391/daily_dialog": "OpenRL/daily_dialog",
    "li2017dailydialog/daily_dialog": "OpenRL/daily_dialog",
    "roskon/dailydialog": "OpenRL/daily_dialog",
}

def load_any_dataset(path_or_name: str, source_type: str, subset_name: str = None):
    # A. Local files / Google Drive files
    if source_type == "Custom Upload / Drive" or os.path.exists(path_or_name):
        ext = os.path.splitext(path_or_name)[-1].lower()
        type_map = {".json": "json", ".jsonl": "json", ".csv": "csv", ".tsv": "csv", ".parquet": "parquet", ".txt": "text"}
        file_type = type_map.get(ext, "text")
        try:
            return load_dataset(file_type, data_files=path_or_name, split="train")
        except Exception:
            ds_dict = load_dataset(file_type, data_files=path_or_name)
            return ds_dict[list(ds_dict.keys())[0]]

    # Parse inline subset if user specified 'repo:subset' or 'repo//subset' in DATASET_PATH
    clean_path = path_or_name.strip()
    clean_subset = str(subset_name).strip() if subset_name and str(subset_name).strip().lower() not in ["none", ""] else None

    if not clean_subset and ":" in clean_path:
        parts = clean_path.split(":", 1)
        clean_path, clean_subset = parts[0].strip(), parts[1].strip()
    elif not clean_subset and "//" in clean_path:
        parts = clean_path.split("//", 1)
        clean_path, clean_subset = parts[0].strip(), parts[1].strip()

    # Auto-redirect known legacy script repositories to canonical Parquet equivalents
    clean_key = clean_path.lower()
    target_repo = KNOWN_PARQUET_MIRRORS.get(clean_key, clean_path)
    if target_repo != clean_path:
        print(f"✓ Detected legacy script repository '{clean_path}'. Auto-redirecting to canonical Parquet dataset: '{target_repo}'")

    def get_best_split(ds_obj):
        if hasattr(ds_obj, "keys"):
            splits = list(ds_obj.keys())
            chosen = next((s for s in splits if "train" in s.lower()), None)
            if not chosen:
                chosen = next((s for s in splits if any(k in s.lower() for k in ["sft", "data", "prompt", "chat"])), None)
            if not chosen:
                chosen = splits[0]
            print(f"✓ Auto-detected split '{chosen}' from available: {splits}")
            return ds_obj[chosen]
        return ds_obj

    # Case 1: User specified a specific subset
    if clean_subset:
        print(f"✓ Requesting dataset subset: '{clean_subset}' from '{target_repo}'")
        try:
            return load_dataset(target_repo, clean_subset, split="train")
        except Exception as e1:
            try:
                ds_obj = load_dataset(target_repo, clean_subset)
                return get_best_split(ds_obj)
            except Exception:
                pass
            print(f"⚠️ Could not load subset '{clean_subset}' directly ({e1}). Checking repository structure...")

    # Case 2: No subset requested OR fallback if requested subset wasn't found directly
    try:
        return load_dataset(target_repo, split="train")
    except Exception as e:
        err_msg = str(e)

        # Handle Hugging Face 'Dataset scripts are no longer supported' error (datasets >= 3.0)
        if "dataset scripts are no longer supported" in err_msg.lower() or "not supported anymore" in err_msg.lower():
            if any(k in clean_key for k in ["daily_dialog", "dailydialog"]):
                print(f"✓ Legacy script blocked by Hugging Face. Auto-redirecting to canonical Parquet dataset: 'OpenRL/daily_dialog'")
                return load_dataset("OpenRL/daily_dialog", split="train")
            raise RuntimeError(
                f"The dataset repository '{path_or_name}' relies on a legacy Python loading script (.py) which modern "
                f"Hugging Face 'datasets >= 3.0' no longer executes for security reasons.\n"
                f"Solution: Please specify a standard Parquet or JSONL dataset repository on Hugging Face."
            ) from e

        # Handle multi-subset configuration requirement (e.g. smoltalk, glue, wikitext)
        if "config" in err_msg.lower() or "builder" in err_msg.lower() or "subset" in err_msg.lower():
            try:
                from datasets import get_dataset_config_names
                cfgs = get_dataset_config_names(target_repo)
                if cfgs:
                    matched_cfg = None
                    if clean_subset:
                        matched_cfg = next((c for c in cfgs if c.lower() == clean_subset.lower()), None)
                    if not matched_cfg:
                        # If user left subset empty, prefer 'all' or 'default' for full dataset, else first config
                        matched_cfg = next((c for c in cfgs if c.lower() in ["all", "default"]), cfgs[0])
                    print(f"✓ Dataset requires a subset. Auto-selected '{matched_cfg}' from available: {cfgs}")
                    try:
                        return load_dataset(target_repo, matched_cfg, split="train")
                    except Exception:
                        ds_obj = load_dataset(target_repo, matched_cfg)
                        return get_best_split(ds_obj)
            except Exception:
                pass

        # Handle missing 'train' split
        try:
            ds_obj = load_dataset(target_repo)
            return get_best_split(ds_obj)
        except Exception:
            pass

        raise e

raw_dataset = load_any_dataset(DATASET_PATH, DATASET_SOURCE, globals().get("DATASET_SUBSET", ""))

if MAX_SAMPLES is not None and len(raw_dataset) > MAX_SAMPLES:
    raw_dataset = raw_dataset.shuffle(seed=42).select(range(MAX_SAMPLES))

detected_domain = detect_domain(set(raw_dataset[0].keys()), raw_dataset[0])
print(f"✓ Ingested {len(raw_dataset):,} samples | Active Domain: '{detected_domain.upper()}'")
print("\n--- Sample Formatted Input Preview ---")
preview_str = format_sample(raw_dataset[0])
print(preview_str[:300] + ("..." if len(preview_str) > 300 else "") + "\n" + "-" * 38)

def tokenize_batch(examples):
    keys = list(examples.keys())
    batch_len = len(examples[keys[0]])
    formatted = []
    for i in range(batch_len):
        row = {k: examples[k][i] for k in keys}
        formatted.append(format_sample(row))

    tokens = tokenizer(
        formatted,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors=None,
    )
    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
    }

tokenized_dataset = raw_dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=1000,
    remove_columns=raw_dataset.column_names,
    desc="Tokenizing for BabyFlash Causal LM",
)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
print(f"✓ Pipeline tokenized {len(tokenized_dataset):,} samples. Tensor shape: {tokenized_dataset[0]['input_ids'].shape}")

In [ ]:
# @title 🧠 Cell 3: Native Qwen2 Architecture Engine (100% GGUF & LM Studio Aligned)
Run_This_Cell= "3" # @param {type:"string"}
import math
import json
import torch
import torch.nn as nn
from transformers import Qwen2Config, Qwen2ForCausalLM

# 1. Standardized Qwen2 Architecture Scale Configurations
SCALE_CONFIGS = {
    "Baby-Nano": {
        "hidden_size": 128,
        "intermediate_size": 384,
        "num_hidden_layers": 4,
        "num_attention_heads": 4,
        "num_key_value_heads": 2,
    },
    "Baby-Tiny": {
        "hidden_size": 256,
        "intermediate_size": 768,
        "num_hidden_layers": 6,
        "num_attention_heads": 8,
        "num_key_value_heads": 2,
    },
    "Baby-Flash": {
        "hidden_size": 384,
        "intermediate_size": 1152,
        "num_hidden_layers": 8,
        "num_attention_heads": 12,
        "num_key_value_heads": 4,
    },
    "Baby-Pro": {
        "hidden_size": 512,
        "intermediate_size": 1536,
        "num_hidden_layers": 12,
        "num_attention_heads": 16,
        "num_key_value_heads": 4,
    },
    "Custom": {
        "hidden_size": CUSTOM_HIDDEN_SIZE,
        "intermediate_size": CUSTOM_HIDDEN_SIZE * 3,
        "num_hidden_layers": CUSTOM_NUM_LAYERS,
        "num_attention_heads": max(4, CUSTOM_HIDDEN_SIZE // 32),
        "num_key_value_heads": max(2, CUSTOM_HIDDEN_SIZE // 64),
    },
}

selected_cfg = SCALE_CONFIGS.get(MODEL_SCALE, SCALE_CONFIGS["Baby-Flash"])

# Ensure attention head divisibility
h_size = selected_cfg["hidden_size"]
n_heads = selected_cfg["num_attention_heads"]
n_kv_heads = selected_cfg["num_key_value_heads"]

if h_size % n_heads != 0:
    for candidate in [16, 12, 8, 4]:
        if h_size % candidate == 0:
            n_heads = candidate
            break
    n_kv_heads = max(2, n_heads // 2)

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
raw_eos = [im_end_id, tokenizer.eos_token_id or 151643]
eos_ids = [x for i, x in enumerate(raw_eos) if x is not None and x not in raw_eos[:i]]

config = Qwen2Config(
    vocab_size=len(tokenizer),
    hidden_size=h_size,
    intermediate_size=selected_cfg["intermediate_size"],
    num_hidden_layers=selected_cfg["num_hidden_layers"],
    num_attention_heads=n_heads,
    num_key_value_heads=n_kv_heads,
    max_position_embeddings=max(MAX_SEQ_LENGTH, 512),
    hidden_act="silu",
    rope_theta=10000.0,
    tie_word_embeddings=True,
    bos_token_id=tokenizer.bos_token_id if tokenizer.bos_token_id is not None else 151643,
    eos_token_id=eos_ids if len(eos_ids) > 1 else eos_ids[0],
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 151643,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Qwen2ForCausalLM(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print(f"✓ Instantiated Native Qwen2 Engine: '{MODEL_SCALE}'")
print(f"✓ Architecture: 'Qwen2ForCausalLM' (Native GGUF / llama.cpp / LM Studio)")
print(f"✓ Active Device: {device.upper()} | Precision: {'FP16/BF16 AMP' if ENABLE_MIXED_PRECISION else 'FP32'}")
print(f"✓ Total Parameters: {total_params:,} ({total_params / 1e6:.2f}M) | Trainable: {trainable_params:,}")
print(f"✓ Vocab Size: {config.vocab_size:,} | Context Length: {config.max_position_embeddings}")
print(f"✓ Layers: {config.num_hidden_layers} | Hidden Dim: {config.hidden_size} | FFN Dim: {config.intermediate_size}")
print(f"✓ Attention Heads: {config.num_attention_heads} Q / {config.num_key_value_heads} KV (Grouped-Query Attention)")
print("=" * 60)

In [ ]:
# @title 🚀 Cell 4: Production Training Engine & GGUF-Ready Model Release
Run_This_Cell= "4" # @param {type:"string"}
import gc
import os
import shutil
import glob
import json
from torch.utils.data import DataLoader
from safetensors.torch import save_file

# 1. Setup Optimizer and Mixed-Precision Scaler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
use_amp = ENABLE_MIXED_PRECISION and (device == "cuda")
if device == "cuda":
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
else:
    scaler = torch.amp.GradScaler("cpu", enabled=False)

train_loader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, shuffle=True)
steps_per_epoch = len(train_loader)
total_target_steps = EPOCHS * (steps_per_epoch // max(1, GRAD_ACCUM_STEPS))

# Cosine Learning Rate Scheduler with Warmup
warmup_steps = max(10, int(0.05 * total_target_steps))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_target_steps), eta_min=1e-6)

# 2. Inspect Existing Drive Checkpoints & Auto-Resume Logic
existing_ckpts = []
if os.path.exists(RUN_CHECKPOINT_DIR):
    for d in os.listdir(RUN_CHECKPOINT_DIR):
        if d.startswith("checkpoint-step-"):
            try:
                s_num = int(d.split("-")[-1])
                existing_ckpts.append((s_num, os.path.join(RUN_CHECKPOINT_DIR, d)))
            except ValueError:
                pass
    existing_ckpts.sort(key=lambda x: x[0])

start_step = 0
start_epoch = 0
loss_history = []

if existing_ckpts:
    latest_step, latest_ckpt_dir = existing_ckpts[-1]
    print("=" * 60)
    print(f"[✓] Existing training session found for Scale '{MODEL_SCALE}' on Dataset '{DATASET_PATH}'.")
    print(f"[✓] Resuming seamlessly from checkpoint: {latest_ckpt_dir} (Step {latest_step:,})...")
    print("=" * 60)

    try:
        weight_file = os.path.join(latest_ckpt_dir, "pytorch_model.bin")
        if os.path.exists(weight_file):
            model.load_state_dict(torch.load(weight_file, map_location=device))
        else:
            model = Qwen2ForCausalLM.from_pretrained(latest_ckpt_dir).to(device)

        state_file = os.path.join(latest_ckpt_dir, "training_state.pt")
        if os.path.exists(state_file):
            state = torch.load(state_file, map_location=device)
            optimizer.load_state_dict(state["optimizer_state_dict"])
            if use_amp and state.get("scaler_state_dict") is not None:
                scaler.load_state_dict(state["scaler_state_dict"])
            start_step = state.get("step", latest_step)
            start_epoch = state.get("epoch", start_step // max(1, steps_per_epoch))
            loss_history = state.get("loss_history", [])

        print(f"✓ Resumed successfully! Continuing from Epoch {start_epoch + 1}, Global Step {start_step:,}...")
    except Exception as e:
        print(f"Notice: Auto-resume encountered issue ({e}). Starting fresh from step 0.")
        start_step = 0
        start_epoch = 0
else:
    print("=" * 60)
    print(f"[+] Starting fresh training run: '{UNIQUE_RUN_ID}' with {total_target_steps:,} optimization steps.")
    print("=" * 60)

# 3. Google Drive Rolling Checkpoint Saver (Resumption State Only, Limit = 3)
def save_drive_checkpoint(curr_step: int, curr_epoch: int, is_emergency: bool = False):
    tag = f"checkpoint-step-{curr_step}" if not is_emergency else f"checkpoint-emergency-step-{curr_step}"
    save_path = os.path.join(RUN_CHECKPOINT_DIR, tag)
    os.makedirs(save_path, exist_ok=True)

    # Save minimal weights for resume
    torch.save(model.state_dict(), os.path.join(save_path, "pytorch_model.bin"))
    config.save_pretrained(save_path)

    # Save Training State (Optimizer, Scaler, Step, Epoch, Loss)
    state = {
        "step": curr_step,
        "epoch": curr_epoch,
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if use_amp else None,
        "loss_history": loss_history,
        "unique_run_id": UNIQUE_RUN_ID,
    }
    torch.save(state, os.path.join(save_path, "training_state.pt"))
    status_label = "EMERGENCY" if is_emergency else "SCHEDULED"
    print(f"\n💾 [{status_label} SYNC] Checkpoint synced to Drive: {save_path}")

    # Enforce Rolling Limit (keep top 3 to protect Drive storage)
    all_ckpts = []
    for d in os.listdir(RUN_CHECKPOINT_DIR):
        if d.startswith("checkpoint-step-"):
            try:
                s_num = int(d.split("-")[-1])
                all_ckpts.append((s_num, os.path.join(RUN_CHECKPOINT_DIR, d)))
            except ValueError:
                pass
    all_ckpts.sort(key=lambda x: x[0])

    while len(all_ckpts) > MAX_CHECKPOINTS_TO_KEEP:
        oldest_step, oldest_dir = all_ckpts.pop(0)
        shutil.rmtree(oldest_dir, ignore_errors=True)
        print(f"🧹 [DRIVE OPTIMIZER] Pruned older checkpoint: {oldest_dir}")

# 4. Final Local Package Exporter (Standard Hugging Face Qwen2 Package)
def export_local_model_package(export_dir: str):
    os.makedirs(export_dir, exist_ok=True)
    print("=" * 60)
    print(f"📦 Packaging Native Qwen2 GGUF-Ready Model into: '{export_dir}'")
    print("=" * 60)

    # A. Save model weights and configuration via Hugging Face standard
    model.save_pretrained(export_dir, safe_serialization=True)
    print(f"  ✓ [safetensors] Saved: model.safetensors")

    # B. Save config.json
    config.save_pretrained(export_dir)
    print(f"  ✓ [config] Saved: config.json (Architecture: 'Qwen2ForCausalLM')")

    # C. Save Tokenizer files & ensure chat_template + eos_token are populated for LM Studio / GGUF
    tokenizer.save_pretrained(export_dir)
    t_cfg_path = os.path.join(export_dir, "tokenizer_config.json")
    if os.path.exists(t_cfg_path):
        with open(t_cfg_path, "r", encoding="utf-8") as f:
            t_cfg = json.load(f)
        if getattr(tokenizer, "chat_template", None):
            t_cfg["chat_template"] = tokenizer.chat_template
        t_cfg["eos_token"] = "<|im_end|>"
        with open(t_cfg_path, "w", encoding="utf-8") as f:
            json.dump(t_cfg, f, indent=2)
    print("  ✓ [tokenizer] Saved: tokenizer.json, tokenizer_config.json, vocab.json, merges.txt (with ChatML template)")

    # D. Save generation_config.json
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    raw_eos = [im_end_id, int(tokenizer.eos_token_id or 151643)]
    clean_eos = [x for i, x in enumerate(raw_eos) if x is not None and x not in raw_eos[:i]]
    gen_config = {
        "bos_token_id": int(config.bos_token_id or 151643),
        "eos_token_id": clean_eos if len(clean_eos) > 1 else clean_eos[0],
        "pad_token_id": int(config.pad_token_id or 151643),
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 50,
        "repetition_penalty": 1.25,
        "max_new_tokens": 128,
        "do_sample": True,
    }
    with open(os.path.join(export_dir, "generation_config.json"), "w", encoding="utf-8") as f:
        json.dump(gen_config, f, indent=2)
    print("  ✓ [generation] Saved: generation_config.json")

    # E. Save Model Card README.md
    model_card = f"""# {MODEL_SCALE} (Native Qwen2 Architecture)

An ultra-efficient, native Qwen2 model fully compatible with **`llama.cpp`**, **LM Studio**, and **Ollama**.

## Architecture Highlights
- **Architecture**: `qwen2` / `Qwen2ForCausalLM` (Native C++ support in `llama.cpp` & LM Studio)
- **Attention**: Grouped-Query Attention (GQA) with RoPE
- **Activation**: SwiGLU / SiLU
- **Vocab Size**: {config.vocab_size:,}
- **Hidden Dim**: {config.hidden_size}
- **FFN Dim**: {config.intermediate_size}

## Quickstart (LM Studio / Ollama)
Run directly in LM Studio:
Drag and drop the converted `.gguf` file directly into LM Studio!
"""
    with open(os.path.join(export_dir, "README.md"), "w", encoding="utf-8") as f:
        f.write(model_card)
    print("  ✓ [model_card] Saved: README.md")
    print(f"🎉 Complete GGUF-aligned release package is ready in: {export_dir}")

# 5. Training Loop with Gradient Accumulation & OOM Guardrails
model.train()
global_step = start_step
accum_step = 0

try:
    for epoch in range(start_epoch, EPOCHS):
        epoch_loss = 0.0
        successful_steps = 0
        optimizer.zero_grad()

        for batch_idx, batch in enumerate(train_loader):
            current_batch_global_step = epoch * steps_per_epoch + batch_idx
            if current_batch_global_step < start_step * GRAD_ACCUM_STEPS:
                continue

            try:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = input_ids.clone()
                if tokenizer.pad_token_id is not None:
                    labels[labels == tokenizer.pad_token_id] = -100

                autocast_device = "cuda" if device == "cuda" else "cpu"
                with torch.amp.autocast(autocast_device, enabled=use_amp):
                    outputs = model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels,
                    )
                    loss = outputs.loss / GRAD_ACCUM_STEPS

                scaler.scale(loss).backward()
                accum_step += 1

                epoch_loss += loss.item() * GRAD_ACCUM_STEPS

                if accum_step % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

                    successful_steps += 1
                    global_step += 1
                    loss_history.append((global_step, round(loss.item() * GRAD_ACCUM_STEPS, 4)))

                    if global_step % 50 == 0 or global_step == 1:
                        current_lr = scheduler.get_last_lr()[0]
                        print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{global_step:>5d}/{total_target_steps}] Loss: {loss.item() * GRAD_ACCUM_STEPS:.4f} | LR: {current_lr:.2e}")

                    if SAVE_TO_DRIVE and (global_step % SAVE_EVERY_N_STEPS == 0):
                        save_drive_checkpoint(global_step, epoch, is_emergency=False)

            except torch.cuda.OutOfMemoryError:
                print(f"⚠️ OOM intercepted at step {global_step}. Purging CUDA cache and resuming...")
                gc.collect()
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                continue

        avg_loss = epoch_loss / max(1, successful_steps * GRAD_ACCUM_STEPS)
        print(f"\n>>> Epoch {epoch+1} Complete | Average Loss = {avg_loss:.4f}\n")

    if SAVE_TO_DRIVE:
        save_drive_checkpoint(global_step, EPOCHS, is_emergency=False)
    export_local_model_package(LOCAL_EXPORT_DIR)
    print(f"🎉 Training fully completed! Final model packaged at: {LOCAL_EXPORT_DIR}")

except KeyboardInterrupt:
    print("\n" + "!" * 60)
    print("⚠️ Training paused by user! Triggering emergency checkpoint...")
    print("!" * 60)
    if SAVE_TO_DRIVE:
        save_drive_checkpoint(global_step, epoch, is_emergency=True)
    export_local_model_package(LOCAL_EXPORT_DIR)

In [ ]:
# @title 💬 Cell 5: Test, Infer & Publish Playground
# @markdown Test generation across domains, or publish your trained model package to Hugging Face Hub.

TEST_PROMPT = "Hello"  # @param {type:"string"}
MAX_NEW_TOKENS = 60  # @param {type:"integer"}
TEMPERATURE = 0.7  # @param {type:"number"}
TOP_P = 0.9  # @param {type:"number"}
TOP_K = 50  # @param {type:"integer"}
REPETITION_PENALTY = 1.25  # @param {type:"number"}

# 1. Interactive Anti-Repetition Inference Function
def test_inference(prompt: str):
    model.eval()
    formatted_prompt = f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    print(f"\n[Input Prompt ({detected_domain.upper()})]: {prompt}")
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs.get("attention_mask", None)

    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    raw_eos = [im_end_id, tokenizer.eos_token_id]
    eos_ids = [x for i, x in enumerate(raw_eos) if x is not None and x not in raw_eos[:i]]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_k=TOP_K,
            top_p=TOP_P,
            repetition_penalty=REPETITION_PENALTY,
            do_sample=(TEMPERATURE > 0),
            eos_token_id=eos_ids if len(eos_ids) > 1 else eos_ids[0],
        )

    prompt_len = input_ids.shape[1]
    new_tokens = output_ids[0][prompt_len:]
    assistant_reply = tokenizer.decode(new_tokens, skip_special_tokens=False)
    if "<|im_end|>" in assistant_reply:
        assistant_reply = assistant_reply.split("<|im_end|>")[0]
    if "<|endoftext|>" in assistant_reply:
        assistant_reply = assistant_reply.split("<|endoftext|>")[0]

    print(f"\n[BabyAI Output]:\n{assistant_reply.strip()}\n")
    return assistant_reply

# Run Main Interactive Test
test_inference(TEST_PROMPT)

# Specialized Domain Verification
if detected_domain == "coding":
    print("--- Domain Code Generation Test ---")
    test_inference("Write a Python function to check if a string is a palindrome.")

elif detected_domain == "mathematics_reasoning":
    print("--- Domain Mathematical Reasoning Test (<think> tag) ---")
    test_inference("Solve for x: 3x + 12 = 36.")

elif detected_domain == "science_stem":
    print("--- Domain Science / STEM Test ---")
    test_inference("Explain Newton's third law of motion in simple terms.")

# 2. 1-Click Hugging Face Hub Publishing (Optional)
if PUSH_TO_HUB and HF_TOKEN and HF_REPO_ID:
    try:
        from huggingface_hub import HfApi, login
        print(f"\nAuthenticating with Hugging Face Hub...")
        login(token=HF_TOKEN)
        api = HfApi()
        print(f"Uploading full '{LOCAL_EXPORT_DIR}' folder to '{HF_REPO_ID}'...")
        api.upload_folder(
            folder_path=LOCAL_EXPORT_DIR,
            repo_id=HF_REPO_ID,
            repo_type="model",
            token=HF_TOKEN
        )
        print(f"🎉 Successfully published to: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"Notice: Hub publish encountered: {e}")
elif PUSH_TO_HUB:
    print("\nNotice: Set PUSH_TO_HUB=True with valid HF_TOKEN and HF_REPO_ID in Cell 1 to publish.")

In [ ]:
# @title 🛠️ Cell 6: Convert Any AI Model Folder or Hugging Face Model to GGUF (llama.cpp)
import os
import sys
import subprocess
from pathlib import Path

format_type = "auto"  # @param ["auto", "f16", "f32", "bf16", "Q8_0", "Q4_K_M", "Q5_K_M", "Q4_K_S", "Q5_K_S", "Q6_K", "Q3_K_M", "Q3_K_S", "Q2_K"]
input_folder_path = "./baby_ai_model"  # @param {type:"string"}
hf_download_folder = ""  # @param {type:"string"}
output_file_path = "./baby_ai_model/baby_ai-Qauto.gguf"  # @param {type:"string"}

input_folder_path = Path(input_folder_path)
output_file_path = Path(output_file_path)
llama_dir = Path("/content/llama.cpp")
build_dir = llama_dir / "build"

def run(command):
    print("$", " ".join(map(str, command)))
    process = subprocess.Popen(
        [str(x) for x in command],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, command)

run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "huggingface_hub"
])

from huggingface_hub import snapshot_download

# --- [EDITED SECTION 1: Smart Path Detection & Downloading] ---
if input_folder_path.is_dir():
    # If user pasted a local path (Colab folder or Google Drive folder like /content/drive/MyDrive/...)
    print(f"📁 Using local model directory: {input_folder_path}")
else:
    # If user provided a Hugging Face Repo ID (e.g. 'BossDefender/Noema-2B-uncensored')
    repo_id = str(input_folder_path).strip()
    model_name = repo_id.split("/")[-1] if "/" in repo_id else repo_id

    # Custom download directory, or auto default to /content/<model_name>
    if hf_download_folder.strip():
        target_dir = Path(hf_download_folder.strip())
    else:
        target_dir = Path(f"/content/{model_name}")

    print(f"📥 Downloading Hugging Face model '{repo_id}' into: {target_dir}")
    target_dir.mkdir(parents=True, exist_ok=True)

    downloaded_path = snapshot_download(
        repo_id=repo_id,
        local_dir=str(target_dir),
        repo_type="model"
    )
    input_folder_path = Path(downloaded_path)

if not input_folder_path.is_dir():
    raise FileNotFoundError(f"Input directory not found: {input_folder_path}")
# -------------------------------------------------------------

if not (llama_dir / ".git").exists():
    run([
        "git", "clone", "--depth", "1",
        "https://github.com/ggml-org/llama.cpp",
        llama_dir
    ])
else:
    run(["git", "-C", llama_dir, "pull", "--ff-only"])

run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", llama_dir / "requirements.txt"
])

run([
    sys.executable, "-m", "pip", "install", "-q",
    "--upgrade", "transformers", "safetensors"
])

run([
    "cmake", "-S", llama_dir, "-B", build_dir,
    "-DCMAKE_BUILD_TYPE=Release",
    "-DLLAMA_BUILD_TOOLS=ON",
    "-DLLAMA_BUILD_EXAMPLES=OFF"
])

run([
    "cmake", "--build", build_dir,
    "--config", "Release",
    "--target", "llama-quantize",
    "--parallel", str(min(4, os.cpu_count() or 2))
])

def run_conversion(command):
    print("$", " ".join(map(str, command)))

    result = subprocess.run(
        [str(x) for x in command],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout, end="")

    if result.returncode == 0:
        return

    missing_mtp_error = (
        "conversion/qwen.py" in result.stdout
        and "opt_num_mtp_layers" in result.stdout
    )

    if missing_mtp_error:
        print("ℹ️ No usable MTP head found; retrying with --no-mtp.")
        run([*command, "--no-mtp"])
        return

    raise subprocess.CalledProcessError(
        result.returncode,
        [str(x) for x in command],
        output=result.stdout,
    )

convert_script = llama_dir / "convert_hf_to_gguf.py"
quantizer = build_dir / "bin" / "llama-quantize"

format_type = format_type.upper()
direct_formats = {"AUTO", "F16", "F32", "BF16", "Q8_0"}

if format_type in direct_formats:
    run_conversion([
        sys.executable,
        convert_script,
        str(input_folder_path),
        "--outfile", str(output_file_path),
        "--outtype", format_type.lower(),
    ])
else:
    temp_file = output_file_path.with_name(
        output_file_path.stem + "-auto.gguf"
    )

    try:
        run_conversion([
            sys.executable,
            convert_script,
            str(input_folder_path),
            "--outfile", str(temp_file),
            "--outtype", "auto",
        ])

        run([
            quantizer,
            str(temp_file),
            str(output_file_path),
            format_type.lower()
        ])
    finally:
        if temp_file.exists():
            temp_file.unlink()

print(f"\n✅ Done! GGUF saved at:\n{output_file_path}")